In [15]:
# %%
# ============================================
# 1) Imports + paths (SLIDING-WINDOW epoch-level wPLI)
# ============================================

from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import mne

try:
    from mne_connectivity import spectral_connectivity_epochs
    CONNECTIVITY_BACKEND = "mne_connectivity"
except Exception:
    from mne.connectivity import spectral_connectivity_epochs
    CONNECTIVITY_BACKEND = "mne.connectivity"

print("Connectivity backend:", CONNECTIVITY_BACKEND)

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data"
DERIVED_ROOT = DATA_ROOT / "derived"

MANIFEST_PATH = DERIVED_ROOT / "manifests" / "manifest_spontaneous_validated.csv"
FEATURE_DIR = DERIVED_ROOT / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# NEW outputs (epoch-level via sliding windows)
FEATURES_B_EDGE_EPOCH_PATH = FEATURE_DIR / "features_B_wpli_edges_epoch_sliding.csv"
FEATURES_B_META_EPOCH_PATH = FEATURE_DIR / "features_B_wpli_edges_epoch_sliding_meta.json"
FEATURES_B_NPZ_EPOCH_PATH  = FEATURE_DIR / "features_B_wpli_matrices_epoch_sliding.npz"

CONNECTIVITY_BANDS_HZ = {
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}

REJECT_PTP_UV = 250.0
EYES_KEEP = "closed"
CONNECTIVITY_MODE = "multitaper"
WPLI_METHOD = "wpli"

# Sliding window settings inside each epoch (8s epochs)
# Use 1s windows with 50% overlap: 8s -> 15 windows (approx)
WIN_LEN_SEC = 1.0
WIN_STEP_SEC = 0.5

print("Manifest:", MANIFEST_PATH)
print("Edges CSV:", FEATURES_B_EDGE_EPOCH_PATH)
print("Matrices NPZ:", FEATURES_B_NPZ_EPOCH_PATH)
print("Sliding window:", WIN_LEN_SEC, "sec, step", WIN_STEP_SEC, "sec")

Connectivity backend: mne_connectivity
Manifest: /Users/I743312/Documents/ketamine project/data/derived/manifests/manifest_spontaneous_validated.csv
Edges CSV: /Users/I743312/Documents/ketamine project/data/derived/features/features_B_wpli_edges_epoch_sliding.csv
Matrices NPZ: /Users/I743312/Documents/ketamine project/data/derived/features/features_B_wpli_matrices_epoch_sliding.npz
Sliding window: 1.0 sec, step 0.5 sec


In [16]:
# %%
# ============================================
# 2) Load manifest and filter eyes-closed
# ============================================

manifest = pd.read_csv(MANIFEST_PATH)

required_cols = ["subject_id", "file_path", "eyes", "drug", "recording_number"]
missing = [c for c in required_cols if c not in manifest.columns]
assert len(missing) == 0, f"Manifest missing columns: {missing}"

df = manifest[manifest["eyes"] == EYES_KEEP].copy()
df = df.sort_values(["subject_id", "recording_number"]).reset_index(drop=True)

print("Recordings in EC-only subset:", len(df))
print("Subjects:", df["subject_id"].nunique())
display(df.groupby("drug").size().rename("n_recordings"))
df.head()

Recordings in EC-only subset: 20
Subjects: 10


drug
awake       10
ketamine    10
Name: n_recordings, dtype: int64

,subject_id,date_str,recording_number,eyes,parse_ok,parse_notes,file_path,file_name,parent_dir,drug,drug_source,drug_order_confidence,passes_basic_checks,check_notes
0,210,20161207,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,210_20161207_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
1,210,20161207,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,210_20161207_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
2,219,20161117,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,219_20161117_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN
3,219,20161117,7,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,219_20161117_0007_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,ketamine,order_rule,True,True,NaN
4,249,20161208,3,closed,True,NaN,/Users/I743312/Documents/ketamine project/data...,249_20161208_0003_eyesClosed_afterICA.set,/Users/I743312/Documents/ketamine project/data...,awake,order_rule,True,True,NaN


In [17]:
# %%
# ============================================
# 3) Helpers: epoch QC, connectivity conversion, vectorization, sliding-window epochs
# ============================================

def peak_to_peak_uv(epoch_data_volts: np.ndarray) -> float:
    ptp_per_ch = np.ptp(epoch_data_volts, axis=1)  # volts
    return float(np.max(ptp_per_ch) * 1e6)

def compute_epoch_ptp_uv(epochs: mne.Epochs) -> np.ndarray:
    data = epochs.get_data()  # (n_epochs, n_channels, n_times)
    out = np.zeros(data.shape[0], dtype=float)
    for e in range(data.shape[0]):
        out[e] = peak_to_peak_uv(data[e])
    return out

def reject_epochs_by_ptp(epochs: mne.Epochs, reject_ptp_uv: float) -> tuple[mne.Epochs, np.ndarray, np.ndarray]:
    """
    Returns:
      epochs_clean
      keep_mask (length n_epochs_original)
      ptp_uv_original (length n_epochs_original)
    """
    ptp_uv = compute_epoch_ptp_uv(epochs)
    keep = ptp_uv <= reject_ptp_uv
    return epochs[keep], keep, ptp_uv

def connectivity_to_dense_square(con, n_ch: int) -> np.ndarray:
    try:
        W = con.get_data(output="dense")
    except TypeError:
        W = con.get_data()

    W = np.asarray(W)
    W = np.squeeze(W)

    # If still 3D (freqs), average across last dim
    if W.ndim == 3:
        W = W.mean(axis=-1)

    if W.ndim != 2 or W.shape != (n_ch, n_ch):
        raise RuntimeError(f"Unexpected wPLI dense shape: {W.shape}, expected {(n_ch, n_ch)}")

    W = np.maximum(W, 0.0)
    W = 0.5 * (W + W.T)
    np.fill_diagonal(W, 0.0)
    return W

def upper_triangle_edges(W: np.ndarray) -> np.ndarray:
    n = W.shape[0]
    tri = np.triu_indices(n, k=1)
    return W[tri]

def epoch_to_sliding_windows_epochs(
    epoch_data: np.ndarray,
    info: mne.Info,
    sfreq: float,
    win_len_sec: float,
    win_step_sec: float,
) -> mne.EpochsArray:
    """
    epoch_data: shape (n_channels, n_times) for ONE epoch (volts)
    Returns an EpochsArray where each epoch is one sliding window segment.
    Shape will be (n_windows, n_channels, n_win_times)
    """
    n_ch, n_times = epoch_data.shape
    win_len = int(round(win_len_sec * sfreq))
    step = int(round(win_step_sec * sfreq))

    if win_len <= 0 or step <= 0:
        raise ValueError("win_len_sec and win_step_sec must be > 0")

    if win_len > n_times:
        raise RuntimeError(f"Window length ({win_len} samples) > epoch length ({n_times} samples)")

    starts = np.arange(0, n_times - win_len + 1, step, dtype=int)
    n_win = len(starts)

    if n_win < 3:
        # wPLI across too-few windows will be unstable / degenerate
        raise RuntimeError(f"Too few windows ({n_win}) for wPLI. Increase epoch length or decrease win_len/step.")

    data = np.zeros((n_win, n_ch, win_len), dtype=float)
    for i, s in enumerate(starts):
        data[i] = epoch_data[:, s:s+win_len]

    # Use dummy events
    events = np.c_[np.arange(n_win), np.zeros(n_win, dtype=int), np.ones(n_win, dtype=int)]
    return mne.EpochsArray(data, info=info, events=events, tmin=0.0, verbose="ERROR")

In [18]:
# %%
# ============================================
# 4) Compute wPLI PER EPOCH via sliding windows inside each epoch
# Save:
# - NPZ of epoch-level matrices (keyed by recording + epoch + band)
# - CSV of epoch-level edge-vector features for ML
# ============================================

rows = []
mat_store = {}

for _, r in df.iterrows():
    sid = str(r["subject_id"])
    fp = r["file_path"]
    drug = str(r["drug"])
    eyes = str(r["eyes"])
    recnum = int(r["recording_number"])

    rec_key = f"{sid}__rec{recnum}"

    try:
        epochs = mne.io.read_epochs_eeglab(fp, verbose="ERROR")
        epochs.load_data()

        sfreq = float(epochs.info["sfreq"])
        n_epochs_before = len(epochs)
        n_ch = int(epochs.info["nchan"])
        epoch_len_sec = epochs.get_data().shape[-1] / sfreq

        epochs_clean, keep_mask, ptp_uv_all = reject_epochs_by_ptp(epochs, REJECT_PTP_UV)
        n_epochs_after = len(epochs_clean)

        if n_epochs_after == 0:
            raise RuntimeError("All epochs rejected by peak-to-peak threshold")

        kept_original_idx = np.where(keep_mask)[0].astype(int)

        # Compute wPLI for each kept epoch using sliding-window "mini-epochs"
        data_clean = epochs_clean.get_data()  # (n_epochs_after, n_ch, n_times)

        for i_clean in range(n_epochs_after):
            i_orig = int(kept_original_idx[i_clean])
            epoch_data = data_clean[i_clean]  # (n_ch, n_times)

            # Build sliding windows epochs for this epoch
            win_epochs = epoch_to_sliding_windows_epochs(
                epoch_data=epoch_data,
                info=epochs_clean.info,
                sfreq=sfreq,
                win_len_sec=WIN_LEN_SEC,
                win_step_sec=WIN_STEP_SEC,
            )

            feat_row = {
                "subject_id": sid,
                "drug": drug,
                "eyes": eyes,
                "recording_number": recnum,
                "epoch_index_original": i_orig,
                "epoch_index_within_clean": int(i_clean),
                "file_path": fp,
                "sfreq": sfreq,
                "n_channels": n_ch,
                "epoch_len_sec": epoch_len_sec,
                "ptp_uv": float(ptp_uv_all[i_orig]),
                "n_epochs_before": n_epochs_before,
                "n_epochs_after": n_epochs_after,
                "n_windows": float(len(win_epochs)),
                "win_len_sec": float(WIN_LEN_SEC),
                "win_step_sec": float(WIN_STEP_SEC),
                "extract_ok": True,
                "extract_error": "",
            }

            for band, (fmin, fmax) in CONNECTIVITY_BANDS_HZ.items():
                con = spectral_connectivity_epochs(
                    win_epochs,
                    method=WPLI_METHOD,
                    mode=CONNECTIVITY_MODE,
                    sfreq=sfreq,
                    fmin=float(fmin),
                    fmax=float(fmax),
                    faverage=True,
                    verbose="ERROR",
                )
                W = connectivity_to_dense_square(con, n_ch=n_ch)

                # Store matrix per epoch per band
                mat_store[f"{rec_key}__e{i_orig:04d}__{band}"] = W

                # Vectorize edges
                edges = upper_triangle_edges(W)
                for j, val in enumerate(edges):
                    feat_row[f"{band}_e{j:04d}"] = float(val)

            rows.append(feat_row)

    except Exception as e:
        # One failure row per recording (keeps failures visible)
        rows.append({
            "subject_id": sid,
            "drug": drug,
            "eyes": eyes,
            "recording_number": recnum,
            "epoch_index_original": np.nan,
            "epoch_index_within_clean": np.nan,
            "file_path": fp,
            "sfreq": np.nan,
            "n_channels": np.nan,
            "epoch_len_sec": np.nan,
            "ptp_uv": np.nan,
            "n_epochs_before": np.nan,
            "n_epochs_after": np.nan,
            "n_windows": np.nan,
            "win_len_sec": float(WIN_LEN_SEC),
            "win_step_sec": float(WIN_STEP_SEC),
            "extract_ok": False,
            "extract_error": str(e),
        })

features_B_edges_epoch = pd.DataFrame(rows)

print("Failure rows:", int((~features_B_edges_epoch["extract_ok"]).sum()))
print("Total rows (epochs + failure rows):", len(features_B_edges_epoch))
features_B_edges_epoch.head()

Failure rows: 0
Total rows (epochs + failure rows): 276


,subject_id,drug,eyes,recording_number,epoch_index_original,epoch_index_within_clean,file_path,sfreq,n_channels,epoch_len_sec,...,beta_e1881,beta_e1882,beta_e1883,beta_e1884,beta_e1885,beta_e1886,beta_e1887,beta_e1888,beta_e1889,beta_e1890
0,210,awake,closed,3,0,0,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,...,0.099119,0.121119,0.123166,0.177084,0.126678,0.091512,0.143003,0.124625,0.147337,0.114081
1,210,awake,closed,3,1,1,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,...,0.122141,0.274741,0.282006,0.295688,0.201086,0.227677,0.266174,0.141889,0.096012,0.128732
2,210,awake,closed,3,2,2,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,...,0.165846,0.260614,0.318826,0.309167,0.174736,0.195465,0.265815,0.144592,0.131348,0.170586
3,210,awake,closed,3,3,3,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,...,0.160806,0.254139,0.231561,0.143410,0.199215,0.178292,0.208095,0.229291,0.122635,0.085973
4,210,awake,closed,3,4,4,/Users/I743312/Documents/ketamine project/data...,250.0,62,8.0,...,0.121452,0.175542,0.156912,0.154722,0.199702,0.160538,0.205120,0.244662,0.191758,0.136001


In [19]:
# %%
# ============================================
# 5) Save features + matrices + meta
# ============================================

features_B_edges_epoch.to_csv(FEATURES_B_EDGE_EPOCH_PATH, index=False)
print("Saved edges CSV:", FEATURES_B_EDGE_EPOCH_PATH)
print("Shape:", features_B_edges_epoch.shape)

np.savez_compressed(FEATURES_B_NPZ_EPOCH_PATH, **{k: v.astype(np.float32) for k, v in mat_store.items()})
print("Saved matrices NPZ:", FEATURES_B_NPZ_EPOCH_PATH)
print("Matrices stored:", len(mat_store))

meta = {
    "connectivity_backend": CONNECTIVITY_BACKEND,
    "method": WPLI_METHOD,
    "mode": CONNECTIVITY_MODE,
    "bands_hz": CONNECTIVITY_BANDS_HZ,
    "reject_ptp_uv": REJECT_PTP_UV,
    "eyes_keep": EYES_KEEP,
    "unit_of_analysis": "epoch (wPLI computed across sliding windows within each epoch)",
    "edge_vectorization": "upper triangle, no diagonal",
    "win_len_sec": WIN_LEN_SEC,
    "win_step_sec": WIN_STEP_SEC,
    "edges_csv": str(FEATURES_B_EDGE_EPOCH_PATH),
    "matrices_npz": str(FEATURES_B_NPZ_EPOCH_PATH),
}
with open(FEATURES_B_META_EPOCH_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("Saved meta:", FEATURES_B_META_EPOCH_PATH)

Saved edges CSV: /Users/I743312/Documents/ketamine project/data/derived/features/features_B_wpli_edges_epoch_sliding.csv
Shape: (276, 5691)
Saved matrices NPZ: /Users/I743312/Documents/ketamine project/data/derived/features/features_B_wpli_matrices_epoch_sliding.npz
Matrices stored: 828
Saved meta: /Users/I743312/Documents/ketamine project/data/derived/features/features_B_wpli_edges_epoch_sliding_meta.json


In [20]:
# %%
# ============================================
# 6) Diagnostic
# ============================================

ok = features_B_edges_epoch[features_B_edges_epoch["extract_ok"] == True].copy()

edge_cols = [c for c in ok.columns if c.startswith("theta_e") or c.startswith("alpha_e") or c.startswith("beta_e")]
print("Edge feature columns:", len(edge_cols))

# Sample a few edge columns to check variance quickly
sample_cols = edge_cols[:10] if len(edge_cols) >= 10 else edge_cols
if len(sample_cols) > 0:
    print(ok[sample_cols].describe().loc[["mean","std","min","max"]])
else:
    print("No edge columns found (unexpected).")

Edge feature columns: 5673
      theta_e0000  theta_e0001  theta_e0002  theta_e0003  theta_e0004  \
mean     0.257484     0.248745     0.246906     0.272332     0.255497   
std      0.105966     0.094566     0.106264     0.109662     0.102682   
min      0.042388     0.027678     0.032108     0.047741     0.029175   
max      0.476607     0.457936     0.491330     0.485642     0.466804   

      theta_e0005  theta_e0006  theta_e0007  theta_e0008  theta_e0009  
mean     0.248657     0.249907     0.257060     0.252313     0.259382  
std      0.096101     0.107778     0.106107     0.114059     0.103602  
min      0.039167     0.049241     0.029333     0.042908     0.036206  
max      0.457570     0.481248     0.496451     0.492897     0.487049  
